# SurfCast SD — EDA on Training Set
**Dataset:** `data/processed/training_set.csv`  
**Target:** `rating_key` (VERY_POOR / POOR / POOR_TO_FAIR / FAIR / FAIR_TO_GOOD / GOOD / EPIC)  
**Goal:** Understand the data, engineer season features, identify which features matter most.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from astral import LocationInfo
from astral.sun import sun
from datetime import date
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

ROOT = Path('..') 
TRAINING_SET = ROOT / 'data/processed/training_set.csv'
ISD_DIR      = ROOT / 'data/raw/isd_wind'

## 1. Load Data

In [ ]:
df = pd.read_csv(TRAINING_SET, parse_dates=['timestamp_utc'])
df['timestamp_utc'] = pd.to_datetime(df['timestamp_utc'], utc=True)

print(f'Shape: {df.shape}')
print(f'Date range: {df["timestamp_utc"].min()} → {df["timestamp_utc"].max()}')
df.head(3)

## 2. Summary Statistics

In [ ]:
print('=== Numeric features ===')
display(df.describe().round(2))

print('\n=== Missing values ===')
missing = df.isnull().sum()
print(missing[missing > 0])

print('\n=== Target distribution ===')
rating_order = ['VERY_POOR','POOR','POOR_TO_FAIR','FAIR','FAIR_TO_GOOD','GOOD','EPIC']
counts = df['rating_key'].value_counts().reindex(rating_order).dropna()
pct = (counts / len(df) * 100).round(1)
display(pd.DataFrame({'count': counts, 'pct': pct}))

print('\n=== Breaks ===')
print(df['break_id'].value_counts())

In [ ]:
# Target distribution bar chart
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#d62728','#ff7f0e','#ffbb78','#2ca02c','#98df8a','#1f77b4','#aec7e8']
counts.plot(kind='bar', ax=ax, color=colors, edgecolor='white')
ax.set_title('Rating Distribution (all breaks, 2022–2024)', fontsize=13)
ax.set_xlabel('')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=30)
for p in ax.patches:
    ax.annotate(f'{p.get_height()/len(df)*100:.1f}%',
                (p.get_x() + p.get_width()/2, p.get_height()),
                ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

## 3. Feature Engineering
### 3a. Season features via `astral`
Rather than one-hot encoding winter/summer/spring/fall (which creates hard edges at arbitrary boundaries), we compute exact sunrise and sunset times for San Diego and derive:
- `day_length_hours` — continuous season proxy; longer days = summer
- `hours_since_sunrise` — how far into the day each hourly row is, in surf-relevant terms
- `season_sin` / `season_cos` — circular encoding of day-of-year so Dec 31 and Jan 1 are numerically close

In [ ]:
SD = LocationInfo('San Diego', 'USA', 'America/Los_Angeles', 32.7157, -117.1611)

def get_sun_times(d: date) -> tuple:
    """Return (sunrise_hour_utc, sunset_hour_utc, day_length_hours) for a date."""
    s = sun(SD.observer, date=d, tzinfo=SD.timezone)
    rise = s['sunrise'].astimezone(__import__('pytz').utc)
    sset = s['sunset'].astimezone(__import__('pytz').utc)
    day_len = (sset - rise).total_seconds() / 3600
    return rise.hour + rise.minute/60, sset.hour + sset.minute/60, day_len

# Cache by date to avoid recomputing for every row
unique_dates = df['timestamp_utc'].dt.date.unique()
sun_cache = {}
for d in unique_dates:
    try:
        sun_cache[d] = get_sun_times(d)
    except Exception:
        sun_cache[d] = (6.5, 18.5, 12.0)  # fallback

df['_date'] = df['timestamp_utc'].dt.date
df['_hour_utc'] = df['timestamp_utc'].dt.hour + df['timestamp_utc'].dt.minute / 60

df['sunrise_hour_utc']    = df['_date'].map(lambda d: sun_cache[d][0])
df['sunset_hour_utc']     = df['_date'].map(lambda d: sun_cache[d][1])
df['day_length_hours']    = df['_date'].map(lambda d: sun_cache[d][2])
df['hours_since_sunrise'] = (df['_hour_utc'] - df['sunrise_hour_utc']).clip(lower=0)

# Circular day-of-year encoding
df['day_of_year'] = df['timestamp_utc'].dt.day_of_year
df['season_sin']  = np.sin(2 * np.pi * df['day_of_year'] / 365)
df['season_cos']  = np.cos(2 * np.pi * df['day_of_year'] / 365)

df.drop(columns=['_date','_hour_utc'], inplace=True)
print('Season features added.')
df[['timestamp_utc','day_length_hours','hours_since_sunrise','season_sin','season_cos']].head()

### 3b. Temperature & Humidity from KSAN ISD
The KSAN weather station files we already downloaded include temperature (TMP) and dewpoint (DEW). From dewpoint we derive relative humidity using the Magnus formula.

In [ ]:
def parse_isd_weather(year: int) -> pd.DataFrame:
    path = ISD_DIR / f'KSAN_{year}.csv'
    raw = pd.read_csv(path, low_memory=False)
    raw['timestamp_utc'] = pd.to_datetime(raw['DATE'], utc=True, errors='coerce')

    # TMP / DEW format: "+0161,1" → 16.1°C (tenths of degrees, comma separates quality flag)
    def parse_isd_temp(s):
        try:
            val = float(str(s).split(',')[0]) / 10
            return np.nan if val == 999.9 else val
        except:
            return np.nan

    raw['temp_c']  = raw['TMP'].apply(parse_isd_temp)
    raw['dewp_c']  = raw['DEW'].apply(parse_isd_temp)

    # Magnus formula: relative humidity from temp + dewpoint
    def rh(t, td):
        return 100 * np.exp((17.625 * td) / (243.04 + td)) / np.exp((17.625 * t) / (243.04 + t))

    raw['humidity_pct'] = rh(raw['temp_c'], raw['dewp_c']).clip(0, 100)

    out = raw[['timestamp_utc','temp_c','humidity_pct']].dropna(subset=['timestamp_utc'])
    return out.set_index('timestamp_utc').resample('h').mean().reset_index()

weather = pd.concat([parse_isd_weather(y) for y in [2022, 2023, 2024]], ignore_index=True)
weather['timestamp_utc'] = weather['timestamp_utc'].dt.floor('h')
print(f'Weather rows: {len(weather)}')
weather.head(3)

In [ ]:
# Merge weather into training set on hourly timestamp
df['_merge_ts'] = df['timestamp_utc'].dt.floor('h')
df = df.merge(weather, left_on='_merge_ts', right_on='timestamp_utc', how='left', suffixes=('','_w'))
df.drop(columns=['_merge_ts','timestamp_utc_w'], inplace=True, errors='ignore')

print(f'temp_c missing: {df["temp_c"].isnull().sum()}')
print(f'humidity_pct missing: {df["humidity_pct"].isnull().sum()}')
df[['temp_c','humidity_pct']].describe().round(2)

## 4. Feature vs Target Plots
Box plots of each numeric feature split by `rating_key`. Shows visually whether the feature separates the rating classes.

In [ ]:
rating_order = ['VERY_POOR','POOR','POOR_TO_FAIR','FAIR','FAIR_TO_GOOD','GOOD','EPIC']
palette = ['#d62728','#ff7f0e','#ffbb78','#2ca02c','#98df8a','#1f77b4','#aec7e8']

features = [
    ('WVHT',               'Wave Height (m)'),
    ('DPD',                'Dominant Period (s)'),
    ('MWD',                'Mean Wave Direction (°)'),
    ('APD',                'Average Period (s)'),
    ('tide_height_m',      'Tide Height (m)'),
    ('wind_speed_mph',     'Wind Speed (mph)'),
    ('wind_direction_deg', 'Wind Direction (°)'),
    ('day_length_hours',   'Day Length (hours)'),
    ('hours_since_sunrise','Hours Since Sunrise'),
    ('temp_c',             'Temperature (°C)'),
    ('humidity_pct',       'Relative Humidity (%)'),
]

fig, axes = plt.subplots(4, 3, figsize=(18, 20))
axes = axes.flatten()

for i, (col, label) in enumerate(features):
    if col not in df.columns:
        axes[i].set_visible(False)
        continue
    sub = df[df['rating_key'].isin(rating_order)]
    sns.boxplot(
        data=sub, x='rating_key', y=col,
        order=rating_order, palette=palette,
        ax=axes[i], showfliers=False
    )
    axes[i].set_title(label, fontsize=11)
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=35, labelsize=8)

for j in range(len(features), len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Each Feature vs Surfline Rating', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

## 5. Season Feature Deep Dive
Season is critical because it controls when dawn happens, which determines how long the offshore morning window lasts. In summer, dawn at ~5:30am gives 4+ hours of clean surf; in winter dawn at ~6:30am shrinks that window.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Day length by month
df['month'] = df['timestamp_utc'].dt.month
monthly_daylen = df.groupby('month')['day_length_hours'].mean()
axes[0].bar(monthly_daylen.index, monthly_daylen.values, color='#1f77b4')
axes[0].set_title('Average Day Length by Month')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Day Length (hours)')
axes[0].set_xticks(range(1,13))

# Rating distribution by season (using day_length proxy)
df['season_label'] = pd.cut(
    df['month'],
    bins=[0,2,5,8,11,12],
    labels=['Winter','Spring','Summer','Fall','Winter2']
)
df['season_label'] = df['season_label'].astype(str).replace('Winter2','Winter')

season_rating = df.groupby(['season_label','rating_key']).size().unstack(fill_value=0)
season_rating = season_rating.div(season_rating.sum(axis=1), axis=0)
good_cols = [c for c in ['FAIR_TO_GOOD','GOOD','EPIC'] if c in season_rating.columns]
season_rating[good_cols].plot(kind='bar', ax=axes[1], colormap='Blues')
axes[1].set_title('FAIR_TO_GOOD+ Ratings by Season')
axes[1].set_xlabel('')
axes[1].set_ylabel('Fraction of Hours')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(fontsize=8)

# Hours since sunrise vs rating_value
sns.boxplot(
    data=df[df['rating_key'].isin(rating_order)],
    x='rating_key', y='hours_since_sunrise',
    order=rating_order, palette=palette,
    ax=axes[2], showfliers=False
)
axes[2].set_title('Hours Since Sunrise vs Rating')
axes[2].set_xlabel('')
axes[2].tick_params(axis='x', rotation=35, labelsize=8)

plt.suptitle('Season Feature Analysis', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Clustering Plot (PCA)
Reduce all numeric features to 2 dimensions with PCA. If features separate the rating classes well, we'll see visible clusters colored by rating.

In [ ]:
feature_cols = [
    'WVHT','DPD','MWD','APD',
    'tide_height_m','wind_speed_mph','wind_direction_deg',
    'day_length_hours','hours_since_sunrise','season_sin','season_cos',
    'temp_c','humidity_pct'
]
feature_cols = [c for c in feature_cols if c in df.columns]

# Sample for speed (PCA on 300k rows is slow)
sample = df[df['rating_key'].isin(rating_order)].dropna(subset=feature_cols).sample(n=15000, random_state=42)

X = StandardScaler().fit_transform(sample[feature_cols])
pca = PCA(n_components=2, random_state=42)
components = pca.fit_transform(X)

pca_df = pd.DataFrame({'PC1': components[:,0], 'PC2': components[:,1], 'rating': sample['rating_key'].values})

fig, ax = plt.subplots(figsize=(10, 7))
color_map = {
    'VERY_POOR':'#d62728','POOR':'#ff7f0e','POOR_TO_FAIR':'#ffbb78',
    'FAIR':'#2ca02c','FAIR_TO_GOOD':'#98df8a','GOOD':'#1f77b4','EPIC':'#9467bd'
}
for rating in rating_order:
    sub = pca_df[pca_df['rating'] == rating]
    if len(sub) == 0:
        continue
    ax.scatter(sub['PC1'], sub['PC2'],
               label=rating, alpha=0.4, s=8,
               color=color_map.get(rating,'grey'))

ax.set_title(f'PCA of All Features — Colored by Rating\n(explains {pca.explained_variance_ratio_.sum()*100:.1f}% of variance)', fontsize=12)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.legend(markerscale=3, fontsize=9, title='Rating')
plt.tight_layout()
plt.show()

# PCA loadings — which features drive each component
loadings = pd.DataFrame(pca.components_.T, index=feature_cols, columns=['PC1','PC2'])
print('\nTop PC1 loadings (most influential features in first component):')
print(loadings['PC1'].abs().sort_values(ascending=False).head(6).round(3))

## 7. Feature Importance via P-Values (ANOVA)
For each numeric feature, we run a one-way ANOVA test across rating groups.  
**Null hypothesis:** the feature has the same mean across all rating categories (i.e. it doesn't help distinguish them).  
**Low p-value** → we reject the null → the feature IS significantly different across ratings → it's important.  
**High p-value** → the feature looks the same across ratings → it's likely unimportant.

In [ ]:
results = []
sub = df[df['rating_key'].isin(rating_order)].dropna(subset=feature_cols)

for col in feature_cols:
    groups = [grp[col].dropna().values for _, grp in sub.groupby('rating_key')]
    groups = [g for g in groups if len(g) > 1]
    if len(groups) < 2:
        continue
    f_stat, p_val = stats.f_oneway(*groups)
    results.append({'feature': col, 'F_statistic': f_stat, 'p_value': p_val})

importance_df = pd.DataFrame(results).sort_values('F_statistic', ascending=False).reset_index(drop=True)
importance_df['significant'] = importance_df['p_value'] < 0.05
display(importance_df.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# F-statistic bar chart (higher = more important)
colors_bar = ['#2ca02c' if s else '#d62728' for s in importance_df['significant']]
axes[0].barh(importance_df['feature'][::-1], importance_df['F_statistic'][::-1], color=colors_bar[::-1])
axes[0].set_title('Feature Importance (ANOVA F-statistic)\nGreen = significant (p < 0.05)', fontsize=11)
axes[0].set_xlabel('F-statistic (higher = more important)')

# P-value chart with 0.05 threshold line
axes[1].barh(importance_df['feature'][::-1], importance_df['p_value'][::-1], color=colors_bar[::-1])
axes[1].axvline(0.05, color='red', linestyle='--', label='p = 0.05 threshold')
axes[1].set_title('P-values per Feature\n(left of red line = statistically significant)', fontsize=11)
axes[1].set_xlabel('p-value')
axes[1].legend()

plt.suptitle('Which Features Best Separate the Surf Rating Classes?', fontsize=13)
plt.tight_layout()
plt.show()

print('\nMOST important features:')
print(importance_df.head(3)[['feature','F_statistic','p_value']].to_string(index=False))
print('\nLEAST important features:')
print(importance_df.tail(3)[['feature','F_statistic','p_value']].to_string(index=False))

## 8. Correlation Heatmap

In [ ]:
le = LabelEncoder()
df['rating_encoded'] = le.fit_transform(df['rating_key'].fillna('POOR'))

corr_cols = feature_cols + ['rating_encoded']
corr = df[corr_cols].corr().round(2)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, linewidths=0.5, ax=ax,
            annot_kws={'size': 8})
ax.set_title('Feature Correlation Matrix\n(rating_encoded = target)', fontsize=13)
plt.tight_layout()
plt.show()

## 9. Per-Break Analysis
Each break responds differently to the same conditions. Same swell = different ratings depending on direction and bottom.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, break_id in zip(axes, ['blacks','la_jolla_shores','pb_point']):
    sub = df[df['break_id'] == break_id]
    counts = sub['rating_key'].value_counts().reindex(rating_order).dropna()
    counts.plot(kind='bar', ax=ax, color=palette[:len(counts)], edgecolor='white')
    ax.set_title(break_id.replace('_',' ').title(), fontsize=11)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=35, labelsize=8)
    ax.set_ylabel('Count')

plt.suptitle('Rating Distribution Per Break', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# DPD and WVHT are usually the most important — show per break
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for col_i, feat in enumerate(['DPD','WVHT']):
    for col_j, break_id in enumerate(['blacks','la_jolla_shores','pb_point']):
        ax = axes[col_i][col_j]
        sub = df[(df['break_id'] == break_id) & (df['rating_key'].isin(rating_order))]
        sns.boxplot(data=sub, x='rating_key', y=feat,
                    order=rating_order, palette=palette, ax=ax, showfliers=False)
        ax.set_title(f'{feat} — {break_id.replace("_"," ").title()}', fontsize=10)
        ax.set_xlabel('')
        ax.tick_params(axis='x', rotation=35, labelsize=7)

plt.suptitle('DPD and WVHT vs Rating per Break', fontsize=13)
plt.tight_layout()
plt.show()

## 10. Summary of Findings
Run the cell below to print a clean summary.

In [ ]:
print('='*55)
print('SURFCAST SD — EDA SUMMARY')
print('='*55)
print(f'Total rows:       {len(df):,}')
print(f'Date range:       2022-01-01 → 2024-12-31')
print(f'Breaks:           blacks, la_jolla_shores, pb_point')
print(f'Class imbalance:  FAIR dominates ({df["rating_key"].value_counts(normalize=True).max()*100:.0f}%)')
print()
print('MOST IMPORTANT FEATURES (by ANOVA F-statistic):')
for _, row in importance_df.head(5).iterrows():
    print(f'  {row["feature"]:25s}  F={row["F_statistic"]:8.1f}  p={row["p_value"]:.2e}')
print()
print('LEAST IMPORTANT FEATURES:')
for _, row in importance_df.tail(3).iterrows():
    print(f'  {row["feature"]:25s}  F={row["F_statistic"]:8.1f}  p={row["p_value"]:.2e}')
print()
print('SEASON NOTE: day_length_hours and hours_since_sunrise')
print('  are continuous season proxies computed from astral')
print('  (exact sunrise/sunset for San Diego, no API needed).')
print('='*55)